# BBC News Archive — Topic Classification, Summarization & Key Entity Extraction
### LangChain + local Ollama LLM

This notebook builds a single LangChain pipeline that, for each BBC news article:

| Step | Task | Output column |
|------|------|---------------|
| 2 | Topic classification (Business / Entertainment / Politics / Sport / Tech) | `Detected_Topic` |
| 3 | 2–3 sentence summary | `Summary` |
| 4 | Key entity extraction (people / organizations / locations) | `Key_Entities` |
| 5 | Everything merged back into the DataFrame | full `df` |

---

In [18]:
# Run once. Restart the kernel afterwards if anything was upgraded.
%pip install -q langchain langchain-core langchain-ollama pandas pydantic

Note: you may need to restart the kernel to use updated packages.


c:\Users\risha\OneDrive\Desktop\GenAI\generative-ai-pgp-ji-2026\.venv\Scripts\python.exe: No module named pip


## 0. Imports and configuration

In [19]:
import re
import time
import textwrap
from pathlib import Path
from typing import List, Literal

import pandas as pd
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# ----------------------------- configuration ---------------------------------
OLLAMA_MODEL    = "llama3.2:latest"            # any model you have pulled locally
OLLAMA_BASE_URL = "http://localhost:11434" # default Ollama endpoint
N_ARTICLES      = 30                       # assignment: first 30 articles
MAX_CHARS       = 6000                     # clip very long articles before sending
NUM_CTX         = 8192                     # context window; lower it if RAM is tight
# -----------------------------------------------------------------------------

llm = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,      # deterministic: we want labels, not creativity
    num_ctx=NUM_CTX,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("LangChain LLM configured:", OLLAMA_MODEL)

LangChain LLM configured: llama3.2:latest


In [20]:
# Sanity check — confirms `ollama serve` is running and the model is pulled.
t0 = time.time()
print(llm.invoke("Reply with exactly the word: OK").content.strip())
print(f"round-trip: {time.time() - t0:.1f}s")

OK
round-trip: 10.1s


## Step 1 — Load the dataset

The loader below auto-detects which version of the BBC dataset you have and normalises it to
three columns: `category` (ground truth), `title`, `text`.

In [21]:
SEARCH_DIRS = [Path("data"), Path("."), Path("..")]
CSV_NAMES = [
    "bbc-news-data.csv", "bbc_news_data.csv",
    "bbc-text.csv", "bbc_text.csv",
    "BBC News Train.csv", "bbc-news.csv",
]


def _find_csv():
    for d in SEARCH_DIRS:
        for name in CSV_NAMES:
            p = d / name
            if p.exists():
                return p
    return None


def _find_raw_folder():
    """Fallback: the original `bbc/<category>/001.txt` folder layout."""
    for d in SEARCH_DIRS:
        for root in (d / "bbc", d / "bbc-fulltext" / "bbc", d):
            if root.is_dir():
                subs = [s for s in root.iterdir()
                        if s.is_dir() and s.name.lower() in
                        {"business", "entertainment", "politics", "sport", "tech"}]
                if len(subs) >= 5:
                    return root
    return None


def load_bbc() -> pd.DataFrame:
    csv_path = _find_csv()
    if csv_path is not None:
        # sep=None + python engine lets pandas sniff comma vs tab automatically
        raw = pd.read_csv(csv_path, sep=None, engine="python", encoding="utf-8", on_bad_lines="skip")
        raw.columns = [c.strip().lower() for c in raw.columns]
        text_col = next((c for c in ("content", "text", "article", "body") if c in raw.columns), None)
        cat_col = next((c for c in ("category", "label", "categoryid", "topic") if c in raw.columns), None)
        if text_col is None:
            raise ValueError(f"No text column found in {csv_path}. Columns: {list(raw.columns)}")

        out = pd.DataFrame()
        out["category"] = raw[cat_col].astype(str).str.strip().str.title() if cat_col else "Unknown"
        if "title" in raw.columns:
            out["title"] = raw["title"].astype(str).str.strip()
        else:
            # bbc-text.csv has no title: use the first clause of the article
            out["title"] = raw[text_col].astype(str).str.split(".").str[0].str.slice(0, 90)
        out["text"] = raw[text_col].astype(str).str.strip()
        print(f"Loaded {len(out)} articles from {csv_path}")
        return out.dropna(subset=["text"]).reset_index(drop=True)

    root = _find_raw_folder()
    if root is not None:
        rows = []
        for cat_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            for f in sorted(cat_dir.glob("*.txt")):
                body = f.read_text(encoding="latin-1").strip()
                lines = [l for l in body.split("\n") if l.strip()]
                rows.append({
                    "category": cat_dir.name.title(),
                    "title": lines[0] if lines else f.stem,
                    "text": "\n".join(lines[1:]) if len(lines) > 1 else body,
                })
        print(f"Loaded {len(rows)} articles from folder {root}")
        return pd.DataFrame(rows)

    raise FileNotFoundError(
        "BBC dataset not found. Put bbc-news-data.csv (or bbc-text.csv) in ./data/ "
        "or next to this notebook. See the prerequisites cell at the top."
    )


df_full = load_bbc()
print(df_full.shape)
df_full.head(3)

Loaded 2225 articles from bbc-news-data.csv
(2225, 3)


,category,title,text
0,Business,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from ..."
1,Business,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said the...
2,Business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $90...


In [22]:
print("Ground-truth category distribution in the full dataset:")
print(df_full["category"].value_counts(), "\n")

# ---- Assignment requirement: use the first 30 articles -----------------------
STRATIFIED = False   # set True to instead take 6 per category (a more informative
                     # accuracy check, since head(30) is usually all one category)

if STRATIFIED:
    df = (df_full.groupby("category", group_keys=False)
                 .head(N_ARTICLES // df_full["category"].nunique())
                 .reset_index(drop=True))
else:
    df = df_full.head(N_ARTICLES).copy().reset_index(drop=True)

print(f"Working set: {df.shape[0]} articles")
df.head()

Ground-truth category distribution in the full dataset:
category
Sport            511
Business         510
Politics         417
Tech             401
Entertainment    386
Name: count, dtype: int64 

Working set: 30 articles


,category,title,text
0,Business,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from ..."
1,Business,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said the...
2,Business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $90...
3,Business,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months t...
4,Business,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover ...


## Step 2 — Topic classification chain

Three things make this reliable rather than "hope the model behaves":

1. **A constrained schema.** `Literal[...]` in the Pydantic model becomes a JSON schema `enum`, which
   Ollama enforces at decode time via `with_structured_output()`. The model physically cannot emit
   a label outside the five categories.
2. **Few-shot examples** — two short worked examples as `human` / `ai` message pairs, exactly as the
   assignment suggests, so the model sees the expected input→output shape.
3. **`temperature=0`** so the label is stable across runs.

In [23]:
CATEGORIES = ["Business", "Entertainment", "Politics", "Sport", "Tech"]


class TopicLabel(BaseModel):
    """The single best-matching news category for an article."""
    category: Literal["Business", "Entertainment", "Politics", "Sport", "Tech"] = Field(
        description="Exactly one category label that best describes the article."
    )


CLASSIFY_SYSTEM = (
    "You are an experienced BBC news desk editor. "
    "Analyze the news article and identify its topic as exactly ONE of the following categories: "
    "Business, Entertainment, Politics, Sport, or Tech.\n"
    "Guidance:\n"
    "- Business: companies, markets, economy, earnings, trade, jobs.\n"
    "- Entertainment: film, music, TV, celebrities, awards, books, art.\n"
    "- Politics: governments, elections, parties, legislation, ministers.\n"
    "- Sport: matches, athletes, leagues, tournaments, transfers.\n"
    "- Tech: consumer technology, software, internet, gadgets, telecoms, gaming hardware.\n"
    "Choose the dominant theme, not a topic that is only mentioned in passing. "
    "Return the label only — no explanation."
)

# Few-shot examples. Braces are doubled so the template engine treats them as literal JSON.
FEW_SHOT = [
    ("human", "Article:\nShares in the airline slid 4% after it warned that fuel costs and weak "
              "transatlantic demand would push full-year profits below analysts' forecasts."),
    ("ai", '{{"category": "Business"}}'),
    ("human", "Article:\nThe defending champions came from two goals down to win 3-2 at home, with "
              "the striker completing his hat-trick in stoppage time to move the club top of the table."),
    ("ai", '{{"category": "Sport"}}'),
    ("human", "Article:\nThe handset maker unveiled a phone with a built-in camera and music player, "
              "saying broadband subscribers now expect to download songs directly to their devices."),
    ("ai", '{{"category": "Tech"}}'),
]

classification_prompt = ChatPromptTemplate.from_messages(
    [("system", CLASSIFY_SYSTEM)] + FEW_SHOT + [("human", "Article:\n{article}")]
)

# LCEL chain: prompt -> schema-constrained LLM -> TopicLabel object
classification_chain = classification_prompt | llm.with_structured_output(TopicLabel)

print(classification_chain)

first=ChatPromptTemplate(input_variables=['article'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an experienced BBC news desk editor. Analyze the news article and identify its topic as exactly ONE of the following categories: Business, Entertainment, Politics, Sport, or Tech.\nGuidance:\n- Business: companies, markets, economy, earnings, trade, jobs.\n- Entertainment: film, music, TV, celebrities, awards, books, art.\n- Politics: governments, elections, parties, legislation, ministers.\n- Sport: matches, athletes, leagues, tournaments, transfers.\n- Tech: consumer technology, software, internet, gadgets, telecoms, gaming hardware.\nChoose the dominant theme, not a topic that is only mentioned in passing. Return the label only — no explanation.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, part

In [24]:
def clip(text: str, max_chars: int = MAX_CHARS) -> str:
    """Keep prompts inside the context window."""
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= max_chars else text[:max_chars] + " ..."


# ---- Step 2 expected output: works on a sample datapoint ---------------------
sample_idx = 0
sample = df.iloc[sample_idx]

print("TITLE :", sample["title"])
print("TEXT  :", textwrap.shorten(sample["text"], 300), "\n")

result = classification_chain.invoke({"article": clip(sample["text"])})
print("Detected topic :", result.category)
print("Actual label   :", sample["category"])

TITLE : Ad sales boost Time Warner profit
TEXT  : Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier. The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner [...] 



Detected topic : Business
Actual label   : Business


## Step 3 — Summarization chain

A plain `prompt | llm | StrOutputParser()` chain. The prompt pins down length (2–3 sentences),
coverage (who / what / when / where / why) and tone (no commentary, no preamble).

In [25]:
SUMMARY_SYSTEM = (
    "You are a news summarizer. Summarize the main points of the news article in 2-3 sentences.\n"
    "Rules:\n"
    "- Cover who, what, when, where and why, as applicable.\n"
    "- Use only facts stated in the article. Never add opinion, speculation or commentary.\n"
    "- Write plain prose in the third person.\n"
    "- Output the summary text only: no preamble, no bullet points, no headings."
)

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", SUMMARY_SYSTEM),
    ("human", "News article:\n\n{article}\n\nSummary (2-3 sentences):"),
])

summarization_chain = summary_prompt | llm | StrOutputParser()

In [26]:
# ---- Step 3 expected output: works on a sample datapoint --------------------
summary = summarization_chain.invoke({"article": clip(sample["text"])}).strip()
print("TITLE:", sample["title"], "\n")
print("SUMMARY:\n", textwrap.fill(summary, 100))

TITLE: Ad sales boost Time Warner profit 

SUMMARY:
 TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet
connections and higher advert sales, with the company benefiting from its investment in Google. The
firm's internet business, AOL, saw a 2% increase in sales but lost 464,000 subscribers in the fourth
quarter, with the company aiming to increase subscribers by offering the service for free to
TimeWarner internet customers. TimeWarner's full-year profit rose 27% to $3.36bn, with the company
projecting 5% operating earnings growth and higher revenue and profit margins for 2005.


## Step 4 — Key entity extraction

Entities are returned as three typed lists rather than a free-text blob, so downstream code can
actually use them. The schema is enforced the same way as the classifier.

In [27]:
class KeyEntities(BaseModel):
    """Named entities explicitly mentioned in the article."""
    people: List[str] = Field(default_factory=list,
                              description="Full names of notable people mentioned.")
    organizations: List[str] = Field(default_factory=list,
                                     description="Companies, teams, agencies, parties, institutions.")
    locations: List[str] = Field(default_factory=list,
                                 description="Countries, cities, regions, venues.")


ENTITY_SYSTEM = (
    "You are an information extraction system. From the article, list the names of any important "
    "people, organizations and places mentioned.\n"
    "Rules:\n"
    "- Extract only entities that literally appear in the article; never invent any.\n"
    "- Use the fullest form of each name given in the text (e.g. 'Tony Blair', not 'Blair').\n"
    "- No duplicates. Leave a list empty if that entity type does not appear.\n"
    "- Do not include dates, money amounts, job titles or generic nouns."
)

entity_prompt = ChatPromptTemplate.from_messages([
    ("system", ENTITY_SYSTEM),
    ("human", "News article:\n\n{article}\n\nExtract the entities."),
])

entity_chain = entity_prompt | llm.with_structured_output(KeyEntities)

In [28]:
# ---- Step 4 expected output: works on a sample datapoint --------------------
ents = entity_chain.invoke({"article": clip(sample["text"])})

print("People        :", ents.people)
print("Organizations :", ents.organizations)
print("Locations     :", ents.locations)

KeyboardInterrupt: 

## Step 5 — Run all three tasks over every article and update the DataFrame

The three chains are composed into a single `RunnableParallel`, so one `.invoke()` per article
returns the topic, summary and entities together. `analyze_article()` adds retries and a
per-task fallback, so a single malformed generation can't lose a whole row.

In [ ]:
# One composed chain: same input dict fans out to all three tasks.
analysis_chain = RunnableParallel(
    topic=classification_chain,
    summary=summarization_chain,
    entities=entity_chain,
)


def _entities_to_string(ents: KeyEntities) -> str:
    parts = []
    if ents.people:
        parts.append("People: " + ", ".join(ents.people))
    if ents.organizations:
        parts.append("Organizations: " + ", ".join(ents.organizations))
    if ents.locations:
        parts.append("Locations: " + ", ".join(ents.locations))
    return " | ".join(parts) if parts else "None found"


def _retry(chain, payload, attempts=2, default=None):
    for i in range(attempts):
        try:
            return chain.invoke(payload)
        except Exception as e:
            if i == attempts - 1:
                print(f"    ! {type(e).__name__}: {e}")
                return default
            time.sleep(1)


def analyze_article(text: str) -> dict:
    """Run classification + summarization + entity extraction on one article."""
    payload = {"article": clip(text)}
    try:
        out = analysis_chain.invoke(payload)          # fast path: one parallel call
    except Exception:
        out = {                                       # fallback: run tasks independently
            "topic": _retry(classification_chain, payload),
            "summary": _retry(summarization_chain, payload, default=""),
            "entities": _retry(entity_chain, payload, default=KeyEntities()),
        }

    topic = out.get("topic")
    ents = out.get("entities") or KeyEntities()
    return {
        "Detected_Topic": topic.category if topic else "Unknown",
        "Summary": (out.get("summary") or "").strip(),
        "Key_Entities": _entities_to_string(ents),
        "People": ents.people,
        "Organizations": ents.organizations,
        "Locations": ents.locations,
    }

In [ ]:
# ---- Iterate over every row (~30 x 3 LLM calls; a few minutes on a local model)
records = []
t_start = time.time()

for i, row in df.iterrows():
    t0 = time.time()
    rec = analyze_article(row["text"])
    records.append(rec)
    print(f"[{i + 1:>2}/{len(df)}] {rec['Detected_Topic']:<13} "
          f"({time.time() - t0:>4.1f}s)  {str(row['title'])[:60]}")

print(f"\nDone in {(time.time() - t_start) / 60:.1f} min")

[ 1/30] Business      (29.2s)  Ad sales boost Time Warner profit
[ 2/30] Business      (61.3s)  Dollar gains on Greenspan speech
[ 3/30] Politics      (56.5s)  Yukos unit buyer faces loan claim
[ 4/30] Business      (74.5s)  High fuel prices hit BA's profits
[ 5/30] Business      (68.4s)  Pernod takeover talk lifts Domecq
[ 6/30] Business      (41.5s)  Japan narrowly escapes recession
[ 7/30] Business      (51.6s)  Jobs growth still slow in the US
[ 8/30] Politics      (66.0s)  India calls for fair trade rules
[ 9/30] Business      (49.6s)  Ethiopia's crop production up 24%
[10/30] Politics      (50.3s)  Court rejects $280bn tobacco case
[11/30] Business      (42.4s)  Ask Jeeves tips online ad revival
[12/30] Business      (59.8s)  Indonesians face fuel price rise
[13/30] Business      (61.6s)  Peugeot deal boosts Mitsubishi
[14/30] Business      (84.8s)  Telegraph newspapers axe 90 jobs
[15/30] Politics      (98.8s)  Air passengers win new EU rights
[16/30] Business      (46.9s)  Chin

In [ ]:
# ---- New columns only -------------------------------------------------------
results_df = pd.DataFrame(records, index=df.index)
results_df[["Detected_Topic", "Summary", "Key_Entities"]].head(10)

,Detected_Topic,Summary,Key_Entities
0,Business,"TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet connections and higher ...","People: Richard Parsons | Organizations: TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission (SE..."
1,Business,The dollar has reached its highest level against the euro in almost three months after Federal Reserve Chairman Alan...,"People: Alan Greenspan, Robert Sinche | Organizations: Federal Reserve, Bank of America, White House | Locations: Ne..."
2,Politics,"The owners of Yukos, Menatep Group, plan to ask Rosneft, the buyer of Yukos' former production unit, to repay a $900...","People: Jamie Firestone, Mikhail Khodorkovsky, Tim Osborne | Organizations: Rosneft, Menatep Group, Yukos, Reuters |..."
3,Business,"British Airways reported a 40% drop in profits to £75m for the three months to December 31, 2004, due to high fuel p...","People: Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul | Organizations: British Airways, Dresdner K..."
4,Business,"Allied Domecq's shares have risen 4% on speculation that France's Pernod Ricard is considering a takeover bid, with ...","Organizations: Pernod Ricard, Wall Street Journal, Financial Times, Diageo, LVMH, Seagram, Glenmorangie, Dunkin' Don..."
5,Business,"Japan's economy experienced a technical recession in the three months to September, with revised figures showing a 0...","People: Heizo Takenaka, Paul Sheard | Organizations: Lehman Brothers | Locations: Japan, Tokyo"
6,Business,"The US Labor Department reported that US firms added 146,000 jobs in January, below market expectations of 190,000 n...","People: Herbert Hoover, Rick Egelton, Ken Mayland, President Bush | Organizations: BMO Financial Group, ClearView Ec..."
7,Politics,"India's finance minister, Palaniappan Chidambaram, criticized the restrictive trade policies of the G7 nations durin...","People: Palaniappan Chidambaram, Gordon Brown | Organizations: United Nations, World Bank, IMF | Locations: India, L..."
8,Business,"Ethiopia's crop production increased by 21% in 2004, reaching 14.27 million tonnes, with good rains, fertilizer use,...","Organizations: Food and Agriculture Organisation, World Food Programme, FAO | Locations: Ethiopia, Eastern Ethiopia,..."
9,Politics,An appeal court in Washington has rejected a US government claim accusing the country's biggest tobacco companies of...,"Organizations: Altria Group, RJ Reynolds Tobacco, Lorillard Tobacco, Liggett Group, Brown and Williamson, Clinton ad..."


In [ ]:
# ---- Final DataFrame: original columns + new columns ------------------------
df_final = pd.concat([df, results_df], axis=1)

print("Columns:", list(df_final.columns))
df_final[["title", "category", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

Columns: ['category', 'title', 'text', 'Detected_Topic', 'Summary', 'Key_Entities', 'People', 'Organizations', 'Locations']


,title,category,Detected_Topic,Summary,Key_Entities
0,Ad sales boost Time Warner profit,Business,Business,"TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet connections and higher ...","People: Richard Parsons | Organizations: TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission (SE..."
1,Dollar gains on Greenspan speech,Business,Business,The dollar has reached its highest level against the euro in almost three months after Federal Reserve Chairman Alan...,"People: Alan Greenspan, Robert Sinche | Organizations: Federal Reserve, Bank of America, White House | Locations: Ne..."
2,Yukos unit buyer faces loan claim,Business,Politics,"The owners of Yukos, Menatep Group, plan to ask Rosneft, the buyer of Yukos' former production unit, to repay a $900...","People: Jamie Firestone, Mikhail Khodorkovsky, Tim Osborne | Organizations: Rosneft, Menatep Group, Yukos, Reuters |..."
3,High fuel prices hit BA's profits,Business,Business,"British Airways reported a 40% drop in profits to £75m for the three months to December 31, 2004, due to high fuel p...","People: Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul | Organizations: British Airways, Dresdner K..."
4,Pernod takeover talk lifts Domecq,Business,Business,"Allied Domecq's shares have risen 4% on speculation that France's Pernod Ricard is considering a takeover bid, with ...","Organizations: Pernod Ricard, Wall Street Journal, Financial Times, Diageo, LVMH, Seagram, Glenmorangie, Dunkin' Don..."
5,Japan narrowly escapes recession,Business,Business,"Japan's economy experienced a technical recession in the three months to September, with revised figures showing a 0...","People: Heizo Takenaka, Paul Sheard | Organizations: Lehman Brothers | Locations: Japan, Tokyo"
6,Jobs growth still slow in the US,Business,Business,"The US Labor Department reported that US firms added 146,000 jobs in January, below market expectations of 190,000 n...","People: Herbert Hoover, Rick Egelton, Ken Mayland, President Bush | Organizations: BMO Financial Group, ClearView Ec..."
7,India calls for fair trade rules,Business,Politics,"India's finance minister, Palaniappan Chidambaram, criticized the restrictive trade policies of the G7 nations durin...","People: Palaniappan Chidambaram, Gordon Brown | Organizations: United Nations, World Bank, IMF | Locations: India, L..."
8,Ethiopia's crop production up 24%,Business,Business,"Ethiopia's crop production increased by 21% in 2004, reaching 14.27 million tonnes, with good rains, fertilizer use,...","Organizations: Food and Agriculture Organisation, World Food Programme, FAO | Locations: Ethiopia, Eastern Ethiopia,..."
9,Court rejects $280bn tobacco case,Business,Politics,An appeal court in Washington has rejected a US government claim accusing the country's biggest tobacco companies of...,"Organizations: Altria Group, RJ Reynolds Tobacco, Lorillard Tobacco, Liggett Group, Brown and Williamson, Clinton ad..."


In [ ]:
df_final.to_csv("bbc_news_analysis_results.csv", index=False)
print("Saved -> bbc_news_analysis_results.csv")
df_final.head()

Saved -> bbc_news_analysis_results.csv


,category,title,text,Detected_Topic,Summary,Key_Entities,People,Organizations,Locations
0,Business,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from ...",Business,"TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet connections and higher ...","People: Richard Parsons | Organizations: TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission (SE...",[Richard Parsons],"[TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission (SEC), Bertelsmann]",[US]
1,Business,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said the...,Business,The dollar has reached its highest level against the euro in almost three months after Federal Reserve Chairman Alan...,"People: Alan Greenspan, Robert Sinche | Organizations: Federal Reserve, Bank of America, White House | Locations: Ne...","[Alan Greenspan, Robert Sinche]","[Federal Reserve, Bank of America, White House]","[New York, London, China]"
2,Business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $90...,Politics,"The owners of Yukos, Menatep Group, plan to ask Rosneft, the buyer of Yukos' former production unit, to repay a $900...","People: Jamie Firestone, Mikhail Khodorkovsky, Tim Osborne | Organizations: Rosneft, Menatep Group, Yukos, Reuters |...","[Jamie Firestone, Mikhail Khodorkovsky, Tim Osborne]","[Rosneft, Menatep Group, Yukos, Reuters]","[Russia, Moscow, US]"
3,Business,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months t...,Business,"British Airways reported a 40% drop in profits to £75m for the three months to December 31, 2004, due to high fuel p...","People: Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul | Organizations: British Airways, Dresdner K...","[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul]","[British Airways, Dresdner Kleinwort Wasserstein, BNP Paribas]",[]
4,Business,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover ...,Business,"Allied Domecq's shares have risen 4% on speculation that France's Pernod Ricard is considering a takeover bid, with ...","Organizations: Pernod Ricard, Wall Street Journal, Financial Times, Diageo, LVMH, Seagram, Glenmorangie, Dunkin' Don...",[],"[Pernod Ricard, Wall Street Journal, Financial Times, Diageo, LVMH, Seagram, Glenmorangie, Dunkin' Donuts, Baskin-Ro...","[UK, France, London, Paris, Scotland]"


## Evaluation — how good is the classifier?

The dataset ships with ground-truth categories, so the classification step can be scored directly.
(The summaries and entities have no gold labels, so those are inspected manually below.)

In [ ]:
correct = (df_final["Detected_Topic"].str.lower() == df_final["category"].str.lower())
print(f"Classification accuracy: {correct.mean():.1%}  ({correct.sum()}/{len(df_final)})\n")

print("Predicted vs actual:")
print(pd.crosstab(df_final["category"], df_final["Detected_Topic"],
                  rownames=["actual"], colnames=["predicted"]), "\n")

mistakes = df_final.loc[~correct, ["title", "category", "Detected_Topic"]]
print("Misclassified articles:" if len(mistakes) else "No misclassifications.")
mistakes

Classification accuracy: 80.0%  (24/30)

Predicted vs actual:
predicted  Business  Politics
actual                       
Business         24         6 

Misclassified articles:


,title,category,Detected_Topic
2,Yukos unit buyer faces loan claim,Business,Politics
7,India calls for fair trade rules,Business,Politics
9,Court rejects $280bn tobacco case,Business,Politics
14,Air passengers win new EU rights,Business,Politics
24,Yukos loses US bankruptcy battle,Business,Politics
28,UK firm faces Venezuelan land row,Business,Politics


In [ ]:
# ---- Readable spot-check of a few full results ------------------------------
for i in range(min(3, len(df_final))):
    r = df_final.iloc[i]
    print("=" * 100)
    print("TITLE    :", r["title"])
    print("ACTUAL   :", r["category"], "   PREDICTED:", r["Detected_Topic"])
    print("SUMMARY  :", textwrap.fill(r["Summary"], 96, subsequent_indent=" " * 11))
    print("ENTITIES :", textwrap.fill(r["Key_Entities"], 96, subsequent_indent=" " * 11))

TITLE    : Ad sales boost Time Warner profit
ACTUAL   : Business    PREDICTED: Business
SUMMARY  : TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by sales of high-speed internet
           connections and higher advert sales, with the company benefiting from its investment
           in Google. The firm's internet business, AOL, saw a 2% increase in sales but lost
           464,000 subscribers in the fourth quarter, with the company aiming to increase
           subscribers by offering the service for free to TimeWarner internet customers.
           TimeWarner's full-year profit rose 27% to $3.36bn, with the company projecting 5%
           operating earnings growth and higher revenue and profit margins for 2005.
ENTITIES : People: Richard Parsons | Organizations: TimeWarner, Google, Warner Bros, AOL, US Securities
           Exchange Commission (SEC), Bertelsmann | Locations: US
TITLE    : Dollar gains on Greenspan speech
ACTUAL   : Business    PREDICTED: Business
SUMM